[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/skyrmions/skyrmions.ipynb)

# Magnetic Skyrmions

This example computes magnetic skyrmions in a thin chiral magnet: a lattice of them crystallizing from a random texture, their fate as the field is lowered and raised, and their drift under a current. In a chiral magnet the crystal has no mirror symmetry, and neighbouring spins prefer to stand a little turned against each other, always in the same sense, while the exchange between them prefers them parallel. Without a field the compromise is a helical spiral; in a field it is a lattice of skyrmions, small whirls in which the spin turns over once, from along the field at the rim to against it at the core. A skyrmion is a knot in the spin texture: its spins point in every direction once, so that no smooth change of the texture undoes it, and the texture's winding, its topological charge, counts the skyrmions. A current through the film carries the whirls along and deflects them sideways, the skyrmion Hall effect.

A spin is modelled as the patch of the plane its moment turns in, intrinsic, and a field as the plane it turns spins in, extrinsic, so that the Zeeman energy is a pairing in plain sight, `-(field & spin)`. The exchange pairs each spin with the field its neighbour exerts, after the neighbour's spin has been carried across the bond by a rotor: the bonds' rotors are a connection, and a rotor turning about its own bond by a twist whose sense is the crystal's handedness, an intrinsic scalar, makes the exchange exactly the Dzyaloshinskii–Moriya interaction, exchange in a frame turned along each bond. Carried around a triangle, the rotors bring a spin back turned, a curvature that no turning of the spins removes, which leaves the texture its whirls. The effective field on a spin is written out as its neighbours' spins carried across the bonds to it, each through the exchange, plus the applied field: the energy is bilinear in the spins, so that sum is the energy's derivative with respect to each spin. Each step turns every spin by a rotor, in its field's plane and, for Gilbert's damping, in that plane's quarter turn about the spin, `turning.commutator(spins.dual())`, so the spins stay unit by construction. A triangle's charge is the holonomy of the texture's own connection, the shortest turns from spin to spin around it.

In the notation of micromagnetics the spins are unit vectors $\mathbf m_i$, and the twisted exchange reads $E = -\sum_{\langle ij\rangle}\big[\cos\theta\,\mathbf m_i\cdot\mathbf m_j + \sin\theta\,\hat{\mathbf r}_{ij}\cdot(\mathbf m_i\times\mathbf m_j) + (1-\cos\theta)(\hat{\mathbf r}_{ij}\cdot\mathbf m_i)(\hat{\mathbf r}_{ij}\cdot\mathbf m_j)\big] - h\sum_i m_{z,i}$ for a twist $\theta$ per bond: the Heisenberg exchange $J = \cos\theta$, the Dzyaloshinskii–Moriya vector $\mathbf D_{ij} = \sin\theta\,\hat{\mathbf r}_{ij}$ of a bulk chiral magnet, whose continuum energy is $D\,\mathbf m\cdot(\nabla\times\mathbf m)$, and an anisotropy of second order. The dynamics is the Landau–Lifshitz–Gilbert equation with the adiabatic spin-transfer torque, $\dot{\mathbf m} = -\gamma\,\mathbf m\times\mathbf H_{\rm eff} + \alpha\,\mathbf m\times\dot{\mathbf m} - (\mathbf u\cdot\nabla)\mathbf m$, with $\mathbf H_{\rm eff} = -\partial E/\partial\mathbf m$, and the charge is $Q = \frac{1}{4\pi}\int\mathbf m\cdot(\partial_x\mathbf m\times\partial_y\mathbf m)\,dx\,dy$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import stack, where
from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext, derivative
from examples.util import shapes
from examples.util.animation import show_states
from examples.quantum.skyrmions import render, scenarios

jax.config.update("jax_enable_x64", True)

mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# A place in the film, a step from its origin, and a bond, a step from site to site; the planes reading a place's run
# across and up the film.
Point = ga.gatype.antivector()                                                # Antivector-
Upright = ga.gatype.from_blades("x y")                                        # Vector+
# A spin: the patch of the plane its moment turns in. A field: the plane it turns spins in.
Spin = ga.gatype.antibivector()                                               # Antibivector-
Field = ga.gatype.bivector()                                                  # Bivector+
Rotor = ga.gatype.rotor()                                                     # Rotor+
# The film's plane, which the applied field turns spins in, and the step off the film.
film = mv.xy                                                                  # [] Field+
off_film = mv.z.dual()                                                        # [] Point-

## The lattice

The film is a triangular lattice of sites, each with six neighbours, which repeats over the rhombus its two periods span and so has no edge: a triangulated grid wrapped into a torus, `shapes.flat_torus`, its triangles equilateral. Where a bond crosses the wrap, no placement of the sites gives its step as the difference of their places, so the lattice comes with its bonds' steps beside its sites' places, each a unit step along one of the lattice's three directions.

In [ ]:
# The lattice's sites along each of its periods.
n_cells = 48

In [ ]:
# The lattice's topology, its sites' places, its bonds' steps and the periods over which it repeats.
topology, coordinates, steps, spans = shapes.flat_torus(n_cells, n_cells)
vertices = mv(Upright, coordinates).dual().field()                            # Point-[vertices]
bonds = mv(Upright, steps).dual().field()                                     # Point-[edges]
periods = mv(Upright, spans).dual()                                           # [2] Point-
render.draw_lattice(topology, vertices, bonds, periods);

The lattice's corner close up: its bonds and sites, and dashed, its periods through the corner, where the lattice wraps; a bond that crosses one reaches the copy of the lattice beyond it.

## Exchange across a twisted bond

The exchange pairs each spin with the field its neighbour exerts on it, the neighbour's spin turned into the plane it is the patch of, `exchange = Spin.dual()`: an isotropic material map, and the one place the metric enters the energy. In a chiral magnet the pairing comes after the neighbour has been carried across the bond by a rotor, as a lattice gauge theory carries its fields: the bonds' rotors are a connection, each carrying the spin at its tail to its head. Each rotor turns about its own bond, `twist * bonds`, by the twist per unit length, an intrinsic scalar whose sense is the crystal's handedness; a mirror reverses it, and with it the sense in which the texture winds. Carried so, the exchange is exactly the Heisenberg exchange, `cos(twist_angle)`, plus the Dzyaloshinskii–Moriya interaction, `sin(twist_angle)`, the plane through two neighbouring spins paired with their bond's twist, plus an anisotropy along the bond of second order in the twist, as the checks confirm. The applied field turns spins in the film's plane and pairs with every spin.

In [ ]:
# The bonds' twist per unit length, in radians.
twist_angle = 0.55

In [ ]:
# Each bond's rotor turns a spin about the bond, by the twist, in the sense of the crystal's handedness.
twist = mv.antipseudoscalar([twist_angle])                                    # [] Scalar-
twists = twist * bonds                                                        # Field+[edges]
carriers = (twists * (-1 / 2)).exp()                                          # Rotor+[edges]
# The exchange, in units of its strength: the field a spin exerts on a neighbour.
exchange = Spin.dual()                                                        # [] Field+ <- Spin-


def energy(applied: Field, spins: Spin) -> Scalar:
    """The energy `[] Scalar+` of the spins `Spin-[vertices]` in the applied field `Field+`: on every bond, the field its
    tail's spin exerts on its head's spin, once the bond's rotor has carried it across, paired with the head's spin;
    and the applied field paired with every spin; each taken negative."""
    tails, heads = topology.incidence[1, 0] * spins                           # Spin-[edges], Spin-[edges]
    exchange_fields = exchange(carriers >> tails)                             # Field+[edges]
    return -(exchange_fields & heads).sites.sum() - (applied & spins).sites.sum()

Carried once around a triangle, the bonds' rotors bring a spin back turned: the holonomy, the product of the rotors of the triangle's sides in the order the triangle runs around them. Were it no turn on any triangle, the twist would be a turning of the spins in disguise, undone by turning each spin by the rotors that carry it there from one site, and the magnet would be a ferromagnet in a turned frame. It is a turn mostly in the film's plane, the same on every triangle, of about the twist squared times the triangle's area: the connection's curvature, which no turning of the spins removes, and which leaves the texture its whirls, on the scale of one over the twist.

In [ ]:
def holonomies(rotors: Rotor) -> Rotor:
    """Each face's holonomy `Rotor+[faces]`: the rotors on the edges `Rotor+[edges]`, each carrying its edge's tail to
    its head, multiplied around the face; it carries a spin at the face's second corner around the face and back."""
    sides = topology.incidence[2, 1] * rotors                                 # [corners] Rotor+[faces]
    # An edge's rotor where the face runs along the edge, its reverse where the face runs against it.
    sides = where(topology.orientation[1].T > 0, sides, ~sides)               # [corners] Rotor+[faces]
    # Around the face, each side carries on from where the one before it arrived, multiplying from the left.
    return sides.cumprod(axis=0)[-1]


# Each triangle's curvature, the logarithm of its holonomy, half the turn it makes.
curvatures = holonomies(carriers).log().batch()                              # [faces] Field+
# The lattice's triangles pointing up come first, then those pointing down.
print("pointing up:\n", curvatures[0].coefficient_table())
print("pointing down:\n", curvatures[n_cells ** 2].coefficient_table())

Half the turn is in the film's plane, `mv.xy`, alike on both kinds of triangle and close to half the twist squared times a triangle's area; the small parts in the other planes, of third order in the twist, depend on the corner the loop starts from.

## The effective field and the step

The effective field on a spin is the field its neighbours exert on it, each neighbour's spin carried across the bond by the bond's rotor, a tail's to the head by `carriers >> tails` and a head's back to the tail by `carriers << heads`, the sandwich that undoes the carrying, plus the applied field. The energy is bilinear in the spins, so this sum is the energy's derivative with respect to each spin, the field that pairs with spins as the energy falls; the checks compare it with JAX's derivative. Each spin precesses in its field's plane, and Gilbert's damping adds that plane's quarter turn about the spin, which turns the spin toward its field; solved for the spin's rate, the Gilbert equation divides the two by `1 + damping ** 2`. A current carries the texture with its electrons' spins, which drift and turn each spin back along its change in their direction, in the plane through the spin and that change, `spins ^ along`. Where a temperature is wanted, a thermal field kicks every spin in every plane. Every spin turns by the rotor of its turning, so it stays unit by construction, and the step is as large as keeps the energy falling. JAX compiles the step and scans the steps between frames.

In [ ]:
# The Gilbert damping, the time step in the exchange's precession time, the steps between frames, and a frame's duration
# in milliseconds.
damping, dt, stride = 0.5, 0.2, 12
frame_duration_ms = 40

In [ ]:
# Each triangle's sides; twice its area, two sides joined with the step off the film; each corner's hat gradient, the
# plane through the facing side and the step off the film over twice the area, which reads one from the side to the
# corner; and each site's share of the area around it.
sides = topology.boundary[2, 1] * bonds                                       # [corners] Point-[faces]
double_areas = (sides[0] & sides[1]) & off_film                               # Scalar-[faces]
gradients = (off_film & sides) / double_areas                                 # [corners] Plane+[faces]
areas = double_areas / 2                                                      # Scalar-[faces]
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
cells = (~corners * (areas / 3)).sum(axis=0)                                  # Scalar-[vertices]


def effective_field(applied: Field, spins: Spin) -> Field:
    """The effective field on each spin `Field+[vertices]`, for the spins `Spin-[vertices]` in the applied field: the
    applied field, plus on every bond at the site the field the spin at the bond's other end exerts, carried across the
    bond."""
    tails, heads = topology.incidence[1, 0] * spins                           # Spin-[edges], Spin-[edges]
    # The field each bond brings to its ends: the other end's spin, carried across the bond, back to the tail by the
    # sandwich that undoes the carrying.
    to_tails = exchange(carriers << heads)                                    # Field+[edges]
    to_heads = exchange(carriers >> tails)                                    # Field+[edges]
    return (~topology.incidence[1, 0] * stack((to_tails, to_heads))).sum(axis=0) + applied


def dynamics(spins: Spin, applied: Field, drifts: Point, time_steps: np.ndarray, temperature: float,
             seed: int) -> Iterator[Spin]:
    """The spins `Spin-[vertices]` at each frame from these, in the applied field `[frames] Field+`, carried by a current
    whose spins drift at the velocity `[frames] Point-`, every step lasting its frame's time step `[frames]`, at the
    temperature, its kicks drawn from the seed."""

    def step(spins: Spin, kicks: jax.Array, applied: Field, drift: Point, time_step: float) -> Spin:
        field = effective_field(applied, spins)                               # Field+[vertices]
        # The thermal field, its spread in each plane the one at which the damping holds the spins at the temperature.
        thermal = (2 * damping * temperature / time_step) ** 0.5 * mv(Field, kicks).field()   # Field+[vertices]
        # Each spin's change along the drift, on each triangle from its corners' hats, shared out to the corners' cells.
        along_faces = ((gradients & drift) * (corners * spins)).sum(axis=0)   # Spin-[faces]
        along = (~corners * (areas / 3 * along_faces)).sum(axis=0) / cells    # Spin-[vertices]
        undamped = field + thermal - (spins ^ along)                          # Field+[vertices]
        # Gilbert's damping adds the turning's quarter turn about the spin.
        turning = (undamped + damping * undamped.commutator(spins.dual())) / (1 + damping ** 2)   # Field+[vertices]
        rotors = (turning * (-time_step / 2)).exp()                           # Rotor+[vertices]
        return rotors >> spins

    # The steps between two frames, compiled and scanned.
    @jax.jit
    def strided(spins: Spin, key: jax.Array, applied: Field, drift: Point, time_step: float) -> Spin:
        kicks = jax.random.normal(key, (stride, topology.n_vertices, 3))
        return jax.lax.scan(lambda spins, kick: (step(spins, kick, applied, drift, time_step), None), spins, kicks)[0]

    keys = jax.random.split(jax.random.PRNGKey(seed), len(applied))
    for frame in range(len(applied)):
        yield spins
        spins = strided(spins, keys[frame], applied[frame], drifts[frame], time_steps[frame])

## Crystallization

From a random texture, smoothed over a site or so, the spins settle in a field well within the skyrmions' range, four tenths of the exchange, with no current and no warmth. The texture changes fastest at first and ever more slowly as the lattice settles, so the time step grows in proportion from frame to frame, from a seventieth of a unit of time to a fifth, and each frame shows about as much change as the last. The twist sets the skyrmions about a dozen lattice spacings apart. The cells below take about ten seconds, most of it drawing the animation.

In [ ]:
# The applied field, in units of the exchange; the frames, and the first frame's time step; the rounds of averaging that
# smooth the random texture; and its seed.
crystal_field, n_crystal_frames, first_time_step = 0.4, 140, 0.014
n_smoothing, seed = 1, 2

In [ ]:
# The random texture, each spin averaged with its neighbours a round; no current, its spins at rest; and each frame's
# time step, growing in proportion from frame to frame.
noise = scenarios.noise(topology, n_smoothing, seed)                          # Spin-[vertices]
at_rest = mv.x.dual() * np.zeros(n_crystal_frames)                            # [frames] Point-
time_steps = np.geomspace(first_time_step, dt, n_crystal_frames)             # [frames]
crystal_states = show_states(dynamics(noise, film * np.full(n_crystal_frames, crystal_field), at_rest, time_steps, 0.0,
                                      seed), render.animate, topology, vertices, bonds, periods,
                             name="skyrmions_crystallizing", duration_ms=frame_duration_ms,
                             colors=render.n_animation_colours)

Each spin's part along the field is its colour, warm where it points with the field, cool against it and grey where it lies in the film, and its part in the film is drawn over the colour in streaks, feathered toward where it points. Over the first frames the noise gathers into worms and islands of spins pointing against the field. Each island rounds into a skyrmion, its spins turning over from the warm rim to the cool core while their part in the film circles the core, every skyrmion in the same sense, the crystal's handedness. The worms shrink or break up into skyrmions, and the skyrmions push each other apart until, in the last frames, they stand in a hexagonal lattice.

## The topological charge

A triangle's charge is the solid angle its three spins enclose, over the whole sphere's. Along each edge the shortest turn from the tail's spin to the head's is half the turn their product makes, `(1 + heads * tails).normalized()`, and these turns are a connection too, the texture's own. Composed around the triangle, they carry the first spin around and back turned about itself by the solid angle, as parallel transport around a triangle of the sphere turns by its area: the holonomy of the texture's connection is its charge, as the holonomy of the bonds' connection is their curvature. Summed over the lattice, which has no edge, the solid angles make whole spheres, one for each skyrmion.

In [ ]:
def charges(spins: Spin) -> Scalar:
    """Each face's topological charge `Scalar+[faces]`, in whole spheres, for the spins `Spin-[vertices]`."""
    tails, heads = topology.incidence[1, 0] * spins                           # Spin-[edges], Spin-[edges]
    # The shortest turn from each edge's tail spin to its head's.
    shortest = (1 + heads * tails).normalized()                               # Rotor+[edges]
    # The holonomy's logarithm, half its turn about the spin at the second corner, in that spin's own plane.
    halves = holonomies(shortest).log()                                       # Field+[faces]
    second = corners[1] * spins                                               # Spin-[faces]
    return -2 * (halves & second) / (4 * np.pi)


crystal = crystal_states[-1]
print(f"the crystal's charge: {charges(crystal).sites.sum().to_array():.6f}")

## The regimes of the field

The same random texture, relaxed in three fields side by side, shows the three textures a chiral magnet takes. With no field the spins wind into the helical spiral, stripes along which they turn over and over at the twist; in the crystal's field they gather into the skyrmion lattice; above the field at which a skyrmion collapses, every island of reversed spins shrinks away, and the ferromagnet remains, every spin along the field. The cells below take about twenty seconds, most of it drawing the animation.

In [ ]:
# The fields the same texture relaxes in: none, the crystal's, and above the skyrmions' collapse.
regime_fields = np.array([0.0, crystal_field, 1.0])

In [ ]:
# The same texture relaxing in each field, the cases side by side, frame by frame.
regime_runs = (dynamics(noise, film * np.full(n_crystal_frames, field), at_rest, time_steps, 0.0, seed)
               for field in regime_fields)
show_states(zip(*regime_runs), render.animate_together, topology, vertices, bonds, periods, name="skyrmions_regimes",
            duration_ms=frame_duration_ms, colors=render.n_animation_colours);

On the left, with no field, the worms of the first frames grow into stripes instead of breaking up into islands, and the stripes meander and slowly straighten. In the middle the islands round into the skyrmion lattice of the crystal above. On the right the islands shrink and vanish within the first frames, and only fading ripples cross the ferromagnet.

## Sweeping the field

The field is lowered, raised past the skyrmions' range, and lowered to nothing, while a little warmth, a fiftieth of the exchange, kicks the spins as they precess. The ferromagnet the high field leaves is a state of rest, from which a texture can grow only out of such kicks. The cells below take about ten seconds, most of it drawing the animation.

In [ ]:
# The frames at which the field passes its turning points, and the field there: lowered, raised until the skyrmions
# vanish, lowered to nothing and held; and the temperature, in units of the exchange.
sweep_frames = np.array([0, 20, 55, 67, 99])
sweep_fields = np.array([0.4, 0.15, 0.95, 0.0, 0.0])
sweep_temperature = 0.02

In [ ]:
fields = np.interp(np.arange(sweep_frames[-1] + 1), sweep_frames, sweep_fields)   # [frames]
still = mv.x.dual() * np.zeros(len(fields))                                   # [frames] Point-
sweep_states = show_states(dynamics(crystal, film * fields, still, np.full(len(fields), dt), sweep_temperature, seed),
                           render.animate, topology, vertices, bonds, periods, name="skyrmions_sweep",
                           duration_ms=frame_duration_ms, colors=render.n_animation_colours)

As the field falls the skyrmions swell until they fill the rectangle, and as it rises they shrink. Near nine tenths of the exchange, a few lattice spacings across, they collapse into the ferromagnet, each taking its unit of charge with it, which only the lattice's discreteness allows: in a continuum the knot would have to shrink to a point. Lowered again, the ferromagnet holds until the field is nearly gone; then the thermal kicks grow into the helical spiral, stripes along which the spins wind at the twist, meandering as a labyrinth. The skyrmions do not come back on the way down: nothing ties the knots again.

## Drift under a current

A current through the film carries the texture along: its electrons' spins follow the local spins and pass on their change, so that the texture moves with their drift where nothing holds it back. The damping drags each skyrmion against its motion, and the skyrmion's winding turns that drag sideways: the lattice drifts at an angle to the current, the skyrmion Hall angle, the larger the larger the damping. The current runs from left to right, switched on after a few frames. The cells below take about ten seconds, most of it drawing the animation.

In [ ]:
# The current's direction, across the picture; the speed at which its spins drift, in lattice spacings per unit of time;
# and the frames before it is switched on and after.
current_direction = mv.x.dual()                                               # [] Point-
drift_speed, n_still_frames, n_driven_frames = 0.2, 10, 90

In [ ]:
speeds = np.concatenate([np.zeros(n_still_frames), np.full(n_driven_frames, drift_speed)])   # [frames]
drive_states = show_states(dynamics(crystal, film * np.full(len(speeds), crystal_field), current_direction * speeds,
                                    np.full(len(speeds), dt), 0.0, seed), render.animate, topology, vertices, bonds,
                           periods, name="skyrmions_driven", duration_ms=frame_duration_ms,
                           colors=render.n_animation_colours)

Once the current is on, the lattice drifts as a whole, every skyrmion alike, down and to the right, at a large angle to the current: with the damping at half the precession's rate, the drag is strong and the Hall angle large. The current carries the skyrmions without unwinding any of them.

In [ ]:
# checks
# Damped, undriven and cold, the energy never rises as the lattice crystallizes.
energies = np.array([energy(crystal_field * film, spins).to_array() for spins in crystal_states])
assert np.all(np.diff(energies) <= 1e-12 * np.abs(energies[0]))
# The written effective field is the energy's derivative with respect to each spin: paired with the spin at its own
# site, by the Zeeman energy's own pairing, and summed over the sites, it is the derivative's form, negated.
pairing = Field & Spin                                                        # [] Scalar+ <- (Field+, Spin-)
applied = crystal_field * film                                                # [] Field+
slope = derivative(lambda spins: energy(applied, spins))(crystal)             # [] Scalar+ <- Spin-[vertices]
# The written field's pairing with the spin at its own site, as a form on the spins at every site.
written = pairing(effective_field(applied, crystal)).on_diagonal().sites.sum()   # [] Scalar+ <- Spin-[vertices]
np.testing.assert_allclose((slope + written).kernel, 0.0, atol=1e-12)
# Every step turns every spin by a rotor: after the sweep and the drive, every spin is still unit.
for states in (sweep_states, drive_states):
    np.testing.assert_allclose(states[-1].norm().batch().to_array(), 1.0, atol=1e-10)
# The crystal's charge is a whole number, one for each skyrmion, a patch of spins pointing against the field.
pointing_down = (film & (corners.mean(axis=0) * crystal)) < 0                 # [faces]
np.testing.assert_allclose(charges(crystal).sites.sum().to_array(),
                           -topology.subset(np.asarray(pointing_down), 2).n_components, atol=1e-9)
# From the field's peak until it is low again, no skyrmion is left: the charge is none.
ferromagnet = sweep_states[sweep_frames[2]:sweep_frames[3]]
np.testing.assert_allclose([charges(spins).sites.sum().to_array() for spins in ferromagnet], 0.0, atol=1e-9)
# The exchange carried across a twisted bond is the Heisenberg exchange, the Dzyaloshinskii-Moriya interaction, the
# plane through the two spins paired with the bond's twist, and an anisotropy along the bond, to round-off.
tails, heads = topology.incidence[1, 0] * crystal                             # Spin-[edges], Spin-[edges]
twist_planes = twists.normalized()                                            # Field+[edges]
heisenberg = -np.cos(twist_angle) * (exchange(tails) & heads).sites.sum()
dzyaloshinskii_moriya = np.sin(twist_angle) * ((heads ^ tails) & twist_planes.dual()).sites.sum()
anisotropy = -(1 - np.cos(twist_angle)) * ((twist_planes & tails) * (twist_planes & heads)).sites.sum()
zeeman = -(crystal_field * film & crystal).sites.sum()
np.testing.assert_allclose(energy(crystal_field * film, crystal).to_array(),
                           (heisenberg + dzyaloshinskii_moriya + anisotropy + zeeman).to_array(), rtol=1e-12)